# Cluster Definition and Back-Projection

For a given CBSA:
1. Load the 2020 dual graph; find the 2 largest connected components of majority-Black tracts
2. Visualize them by centroid scatter and polygon choropleth
3. Back-project both clusters to 1980–2010 via areal overlap (>50% of each earlier-year tract)
4. Check graph connectivity of the back-projected tracts in the earlier-year dual graphs
5. Save all-years cluster membership to CSV in the `manual_cluster_tracts.csv` format

In [ ]:
import json
from pathlib import Path
from shapely.ops import unary_union
import networkx as nx
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

CBSA = "37980" #"16980" = Chicago, "37980" = Philly

BLACK_SHARE_THRESHOLD = 0.53 # Philly's rho = 53%; Chicago's rho = 48%
#tracts with BLACK/TOTPOP > threshold are "majority-Black"

OVERLAP_THRESHOLD = 0.50 # earlier-year tract must have >50% of its area in cluster. maybe this should be changed to a higher value

YEARS = [1980, 1990, 2000, 2010]

ROOT = Path("/Users/maria/Documents/capy-bara")
DUAL_GRAPHS_DIR = ROOT / "data" / "processed" / "dual_graphs"
CLIPPED_GEO_DIR = ROOT / "data" / "processed" / "clipped_geographies"
OUTPUT_FILE = ROOT / "experiments" / "h4_t3_observed_diffusion" / "data" / "auto_cluster_tracts.csv"

COLORS = {"cluster_1": "steelblue", "cluster_2": "tomato"}

## Step 1 — 2020 dual graph: define majority-Black clusters

In [ ]:
# graph_file = DUAL_GRAPHS_DIR / "2020" / f"tracts_in_cbsa_{CBSA}_2020_march_2020_vintage_connected.json"
graph_file = DUAL_GRAPHS_DIR / "2020" / f"tracts_in_cbsa_{CBSA}_2020_march_2020_vintage_orig.json"

with open(graph_file) as f:
    graph_data = json.load(f)

G_2020 = nx.Graph()
for node in graph_data["nodes"]:
    G_2020.add_node(node["id"], **node)
for node_id, neighbors in enumerate(graph_data["adjacency"]):
    for edge in neighbors:
        G_2020.add_edge(node_id, edge["id"], shared_perim=edge["shared_perim"])

print(f"2020 graph: {G_2020.number_of_nodes()} nodes, {G_2020.number_of_edges()} edges")

In [ ]:
nodes_df = pd.DataFrame([G_2020.nodes[n] for n in G_2020.nodes()])
nodes_df["black_share"] = nodes_df["BLACK"] / (nodes_df["BLACK"] + nodes_df["WHITE"]).replace(0, pd.NA) #nodes_df["TOTPOP"].replace(0, pd.NA) # or out of White + Black?
nodes_df["majority_black"] = nodes_df["black_share"] > BLACK_SHARE_THRESHOLD

print(f"Total tracts: {len(nodes_df)}")
print(f"Majority-Black (>{BLACK_SHARE_THRESHOLD:.0%}): {nodes_df['majority_black'].sum()}")
print(f"Below threshold: {(~nodes_df['majority_black']).sum()}")

In [ ]:
majority_ids = set(nodes_df.loc[nodes_df["majority_black"], "id"])
G_black = G_2020.subgraph(majority_ids)

components = sorted(nx.connected_components(G_black), key=len, reverse=True)
print(f"Connected components in majority-Black subgraph: {len(components)}")
print(f"Sizes of top 5: {[len(c) for c in components[:5]]}")

cluster_node_ids = {"cluster_1": components[0], "cluster_2": components[1]}
for label, ids in cluster_node_ids.items():
    print(f"  {label}: {len(ids)} tracts")

In [ ]:
components[5]

In [ ]:
other_black_ids = majority_ids - cluster_node_ids["cluster_1"] - cluster_node_ids["cluster_2"]

fig, ax = plt.subplots(figsize=(10, 10))

ax.scatter(
    nodes_df["centroid_x"], nodes_df["centroid_y"],
    s=4, color="lightgray", alpha=0.5, zorder=1
)
ax.scatter(
    nodes_df.loc[nodes_df["id"].isin(other_black_ids), "centroid_x"],
    nodes_df.loc[nodes_df["id"].isin(other_black_ids), "centroid_y"],
    s=8, color="gold", alpha=0.7, zorder=2, label="Majority-Black (other components)"
)
for label, ids in cluster_node_ids.items():
    sub = nodes_df[nodes_df["id"].isin(ids)]
    ax.scatter(
        sub["centroid_x"], sub["centroid_y"],
        s=14, color=COLORS[label], alpha=0.85, zorder=3,
        label=f"{label} (n={len(ids)})"
    )

ax.set_title(f"CBSA {CBSA} — 2020 majority-Black clusters (centroids)", fontsize=13)
ax.set_xlabel("centroid_x")
ax.set_ylabel("centroid_y")
ax.legend()
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

## Step 2 — Map graph nodes to 2020 polygon geometries

In [ ]:
gpkg_2020 = CLIPPED_GEO_DIR / "2020" / f"tracts_in_cbsa_{CBSA}_2020_march_2020_vintage.gpkg"
gdf_2020 = gpd.read_file(gpkg_2020)

# Map graph node ids → GEOIDs → gpkg rows
cluster_geoids = {label: set(nodes_df.loc[nodes_df["id"].isin(ids), "GEOID"])
    for label, ids in cluster_node_ids.items()}

cluster_gdfs_2020 = {}
for label, geoids in cluster_geoids.items():
    gdf = gdf_2020[gdf_2020["GEOID"].isin(geoids)].copy()
    gdf["cluster"] = label
    cluster_gdfs_2020[label] = gdf
    print(f"2020 {label}: {len(gdf)} tracts matched in gpkg (of {len(geoids)} graph nodes)")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 12))
gdf_2020.plot(ax=ax, color="#e8e8e8", edgecolor="white", linewidth=0.3)
for label, gdf in cluster_gdfs_2020.items():
    gdf.plot(ax=ax, color=COLORS[label], alpha=0.75, edgecolor="white", linewidth=0.3)

legend_handles = [
    mpatches.Patch(facecolor=COLORS[lbl], alpha=0.75,
                   label=f"{lbl} (n={len(cluster_gdfs_2020[lbl])})")
    for lbl in ["cluster_1", "cluster_2"]]
ax.legend(handles=legend_handles, fontsize=11)
ax.set_title(f"CBSA {CBSA} — 2020 majority-Black clusters (polygons)", fontsize=13)
ax.set_axis_off()
plt.tight_layout()
plt.show()

## Step 3 — Back-project to 1980–2010 via areal overlap

For each earlier-year tract: if `intersection_area / tract_area > 0.50`, it belongs to the cluster.

In [ ]:
def back_project_cluster(gdf_cluster_2020, gdf_target, overlap_threshold=0.50):
    """
    Returns rows of gdf_target whose tract area overlaps > overlap_threshold with the dissolved 2020 cluster polygon.
    """
    if gdf_target.crs != gdf_cluster_2020.crs:
        gdf_target = gdf_target.to_crs(gdf_cluster_2020.crs)

    cluster_union = unary_union(gdf_cluster_2020.geometry)

    target = gdf_target.copy()
    target["_tract_area"] = target.geometry.area
    target["_inter_area"] = target.geometry.intersection(cluster_union).area
    target["_overlap"] = target["_inter_area"] / target["_tract_area"]

    return (target[target["_overlap"] > overlap_threshold]
        .drop(columns=["_tract_area", "_inter_area", "_overlap"])
        .copy())

In [ ]:
cluster_gdfs_2020['cluster_1'].head(3)

In [ ]:
# cluster_yearly[year][cluster_label] → GeoDataFrame of matched tracts
cluster_yearly = {2020: cluster_gdfs_2020}

for year in YEARS:
    gpkg_path = CLIPPED_GEO_DIR / str(year) / f"tracts_in_cbsa_{CBSA}_{year}_march_2020_vintage.gpkg"
    gdf_year = gpd.read_file(gpkg_path)
    cluster_yearly[year] = {}

    for label, gdf_cluster in cluster_gdfs_2020.items():
        matched = back_project_cluster(gdf_cluster, gdf_year, OVERLAP_THRESHOLD)
        matched = matched.copy()
        matched["cluster"] = label
        cluster_yearly[year][label] = matched
        print(f"{year} {label}: {len(matched)} tracts")

In [ ]:
all_years = sorted(cluster_yearly.keys())
fig, axes = plt.subplots(1, len(all_years), figsize=(4 * len(all_years), 5))

for ax, year in zip(axes, all_years):
    gpkg_path = CLIPPED_GEO_DIR / str(year) / f"tracts_in_cbsa_{CBSA}_{year}_march_2020_vintage.gpkg"
    base = gpd.read_file(gpkg_path)
    base.plot(ax=ax, color="#e8e8e8", edgecolor="white", linewidth=0.2)

    for label, gdf in cluster_yearly[year].items():
        if len(gdf) > 0:
            gdf.plot(ax=ax, color=COLORS[label], alpha=0.75, edgecolor="white", linewidth=0.2)

    ax.set_title(str(year), fontsize=11)
    ax.set_axis_off()

legend_handles = [
    mpatches.Patch(facecolor=COLORS[lbl], alpha=0.75, label=lbl)
    for lbl in ["cluster_1", "cluster_2"]
]
fig.legend(handles=legend_handles, loc="lower center", ncol=2, fontsize=10,
           bbox_to_anchor=(0.5, -0.02))
fig.suptitle(f"CBSA {CBSA} — clusters across decades (areal overlap >{OVERLAP_THRESHOLD:.0%})",
             fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
base.head(4)
base["BLACK"] / (base["BLACK"] + base["WHITE"])

In [ ]:
all_years = sorted(cluster_yearly.keys())
fig, axes = plt.subplots(1, len(all_years), figsize=(4 * len(all_years), 5))

cmap = "Greens"
vmin, vmax = 0, 1

for ax, year in zip(axes, all_years):
    gpkg_path = CLIPPED_GEO_DIR / str(year) / f"tracts_in_cbsa_{CBSA}_{year}_march_2020_vintage.gpkg"
    base = gpd.read_file(gpkg_path)
    base["black_share"] = base["BLACK"] / (base["BLACK"] + base["WHITE"])

    base.plot(
        ax=ax, column="black_share", cmap=cmap, vmin=vmin, vmax=vmax,
        edgecolor="white", linewidth=0.2,
        missing_kwds={"color": "lightgray", "edgecolor": "white", "linewidth": 0.2})

    cluster_gdfs_proj = []
    for label, gdf in cluster_yearly[year].items():
        if len(gdf) > 0:
            gdf_proj = gdf.to_crs(base.crs)
            gdf_proj.dissolve().boundary.plot(ax=ax, color=COLORS[label], linewidth=2.5, zorder=5)
            cluster_gdfs_proj.append(gdf_proj)

    minx, miny, maxx, maxy = pd.concat(cluster_gdfs_proj).total_bounds
    pad_x = (maxx - minx) * 0.3
    pad_y = (maxy - miny) * 0.3
    ax.set_xlim(minx - pad_x, maxx + pad_x)
    ax.set_ylim(miny - pad_y, maxy + pad_y)

    ax.set_title(str(year), fontsize=11)
    ax.set_axis_off()

# Reserve bottom 30% so colorbar and legend fit without overlapping
fig.tight_layout(rect=[0, 0.30, 1, 1])

sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=vmin, vmax=vmax))
sm.set_array([])
# Colorbar in the upper portion of the reserved strip
cbar_ax = fig.add_axes([0.40, 0.18, 0.20, 0.03])
cbar = fig.colorbar(sm, cax=cbar_ax, orientation="horizontal")
cbar.set_label("Black population share (BLACK / (BLACK + WHITE))", fontsize=10)

# Legend below the colorbar
legend_handles = [
    mpatches.Patch(facecolor="none", edgecolor=COLORS[lbl], linewidth=2.5,
                   label=f"{lbl} boundary")
    for lbl in ["cluster_1", "cluster_2"]]
fig.legend(handles=legend_handles, loc="lower center", ncol=2, fontsize=10,
           bbox_to_anchor=(0.5, 0.03))
plt.show()

## Step 4 — Graph connectivity check for earlier years

Load the earlier-year `_orig.json` dual graphs and check whether the areal-overlap-assigned
tracts form connected subgraphs.

**Why `_orig` not `_connected`**: the `_connected` files have a node id-position mismatch —
the `"id"` field doesn't equal the node's index in the adjacency list. The loading code uses
`enumerate()` for edge endpoints but `node["id"]` for `add_node()`, so `_connected` creates
phantom nodes and wrong edges. `_orig` files have zero mismatches in every year, so they
load correctly. This was also the root cause of the "disconnected-looking cluster_1" in the
scatter plot — with `_connected`, the wrong graph structure merged geographically separate
neighborhoods into one huge component.

In [ ]:
def load_graph(json_path):
    """Load a dual-graph JSON into a networkx Graph. Returns (G, node_df)."""
    with open(json_path) as f:
        data = json.load(f)
    G = nx.Graph()
    for node in data["nodes"]:
        G.add_node(node["id"], **node)
    for node_id, neighbors in enumerate(data["adjacency"]):
        for edge in neighbors:
            G.add_edge(node_id, edge["id"], shared_perim=edge["shared_perim"])
    return G, pd.DataFrame(data["nodes"])

In [ ]:
gdf_matched.head(2)

In [ ]:
connectivity_rows = []

for year in YEARS:
    json_path = DUAL_GRAPHS_DIR / str(year) / f"tracts_in_cbsa_{CBSA}_{year}_march_2020_vintage_orig.json"
    G_year, node_df_year = load_graph(json_path)

    for label in ["cluster_1", "cluster_2"]:
        gdf_matched = cluster_yearly[year][label]
        if len(gdf_matched) == 0:
            continue

        matched_geoids = set(gdf_matched["GEOID"])
        matched_node_ids = set(
            node_df_year.loc[node_df_year["GEOID"].isin(matched_geoids), "id"])
        unmatched = matched_geoids - set(node_df_year["GEOID"])

        G_sub = G_year.subgraph(matched_node_ids)
        comps = sorted(nx.connected_components(G_sub), key=len, reverse=True)

        connectivity_rows.append({
            "year": year,
            "cluster": label,
            "areal_overlap_tracts": len(matched_geoids),
            "in_graph": len(matched_node_ids),
            "not_in_graph": len(unmatched),
            "n_components": len(comps),
            "largest_component": len(comps[0]) if comps else 0
        })

conn_df = pd.DataFrame(connectivity_rows)
print(conn_df.to_string(index=False))

## Step 5 — Assemble and save output CSV

In [ ]:
cluster_yearly[2020]#['cluster_2']
cluster_yearly[2000]['cluster_1'].head(2)

In [ ]:
rows = []

for year, clusters in cluster_yearly.items():
    for label, gdf in clusters.items():
        if len(gdf) == 0:
            raise
        sub = gdf[["GEOID", "GISJOIN", "STATEFP", "COUNTYFP", "BLACK", "WHITE", "POC", "TOTPOP"]].copy()
        sub["cbsa"] = CBSA
        sub["year"] = year
        sub["cluster"] = label
        sub["black_share"] = sub["BLACK"] / (sub["BLACK"] + sub["WHITE"]).replace(0, pd.NA) #sub["TOTPOP"].replace(0, pd.NA)
        rows.append(sub)

df_out = (
    pd.concat(rows, ignore_index=True)
    .rename(columns={"BLACK": "black_population", "TOTPOP": "total_population",
                     "WHITE": "white_population", "POC": "poc_population",
                     "GEOID": "geoid", "GISJOIN": "gisjoin",
                     "STATEFP": "statefp", "COUNTYFP": "countyfp"})
    [["cbsa", "year", "cluster", "gisjoin", "geoid",
      "black_population", "total_population", "black_share"]]
    .sort_values(["cluster", "year", "geoid"])
    .reset_index(drop=True))

df_out.to_csv(OUTPUT_FILE, index=False)
print(f"Saved {len(df_out)} rows to {OUTPUT_FILE}")
df_out.groupby(["year", "cluster"]).size().unstack("cluster")